<a href="https://colab.research.google.com/github/tikoneaayush/ML/blob/main/Used_Car_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

USED CAR PRICE PREDICTION


**TABLE OF CONTENTS**

1. Introduction
2. Problem Statement
3. Business Objective
4. Importing Libraries
5. Loading the Dataset
6. Dataset Structure
7. Data Types
8. Target Variable Analysis
9. Price Distribution Visualization
10. Missing Value Analysis
11. Data Cleaning
12. Feature Understanding
13. Odometer Outlier Removal
14. Feature Engineering
15. Exploratory Data Analysis (EDA)
16. Feature Selection
17. Encoding Categorical Variables
18. Preparing Training and Testing Data
19. Linear Regression Model
20. Decision Tree Regressor
21. Random Forest Regressor
22. Model Evaluation & Comparison
23. Recommended Model

Introduction

We have developed a machine learning based on regression model to predict the price of a used car.
Used car buyers and sellers face challenges in determining the fair market value of a vehicle. Predictive analytics helps reduce pricing uncertainty and improve decision making.

Problem Statement

Determining the correct price of a used car is a major challenge for both buyers and sellers. Lack of pricing transparency leads to overpaying or underselling, causing financial loss on both sides.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/vehicles.csv')

What: Imported core libraries and loaded the used car listings dataset from vehicles.csv into a DataFrame.

Why: Every ML project starts here. Loading into pandas gives structured access to every row and column for exploration, cleaning, and modeling.

Conclusion: Dataset loaded successfully. Foundation ready for all subsequent steps.

In [ ]:
df.head()

What: Displayed first 5 rows including columns like price, year, manufacturer, model, odometer, condition, fuel, region, URL, description.

Why: First visual check — confirms columns loaded correctly, shows data types at a glance, and reveals immediately which columns look useful vs junk (like image_url, VIN).

Conclusion: Dataset contains rich car listing data. Columns like url, image_url, county visible as non-predictive junk to drop later.

In [ ]:
print(df.shape)

What: Printed the exact number of rows and columns in the raw dataset.

Why: Before doing anything you need to know the scale of data you're working with. Shape tells you if you have enough records to train a reliable model.

Conclusion: 18,400 rows × 26 columns confirmed. Sufficient data for ML. 26 columns means aggressive feature selection will be needed.

In [ ]:
print(df.columns.tolist())

What: Printed all 26 column names as a list.

Why: Gives a full inventory of every feature available. Forces you to think about which columns are predictors, which is the target, and which are irrelevant noise.

Conclusion: 26 columns identified. Obvious junk columns spotted immediately — id, url, region_url, image_url, county, VIN, description. Target column is price.

In [ ]:
df.info()

What: Displayed data types and non-null counts for all 26 columns.

Why: Reveals which columns have missing values and what type each column is — essential before any cleaning or encoding decisions.

Conclusion: Multiple columns had heavy nulls — condition, cylinders, drive, paint_color, size, VIN. Mix of object, int64, and float64 dtypes confirmed.


In [ ]:
print(df['price'].describe())

What: Generated count, mean, std, min, max, and quartiles for the price column — the target variable.

Why: Price is what the entire model predicts. Before anything else you must understand its range, spread, and whether it has outliers or impossible values.

Conclusion: Mean price was $79,365 but  max  was $987,654,321 — clearly a data entry error. Min was $0 — impossible for a real car. Massive cleaning needed.

In [ ]:
print(df['price'].skew())

What: Calculated the skewness score of the price column.

Why: Skewness above 1 means the distribution is heavily right-tailed — a few extreme values are pulling the mean far from the median. This hurts linear models significantly.

Conclusion: Skewness of 133.72 — extremely right-skewed. Confirmed that extreme outlier prices (like $987M) are massively distorting the distribution. Cleaning is urgent.


In [ ]:
print("Price <= 0:", (df['price'] <= 0).sum())

print("Price > 100000:", (df['price'] > 100000).sum())

What: Counted how many rows have prices at or below $0 and above $100,000.

Why: Prices of $0 are clearly errors — no car is free. Prices above $100K on a used car platform are almost certainly data entry errors or fake listings.

Conclusion: 939 rows with price ≤ 0 (impossible). 32 rows above $100,000 (likely junk). Total of 971 rows flagged for removal.


In [ ]:
df[['price']].sort_values(
    'price',
    ascending=False
).head(20)

What: Displayed the 20 highest-priced listings sorted descending.

Why: Seeing the actual outlier values confirms they are garbage data — $987,654,321 and $99,999,999 are clearly fake placeholder values, not real car prices.

Conclusion: Top 2 values ($987M and $99.9M) are definitively fake. Even $239,995 is suspicious for a used car platform. Confirmed the $100,000 cap as the right cleaning threshold.


We're trying to answer:

"What range contains realistic vehicle prices?"

What: Markdown cell defining the business question that guides the next cleaning decision.

Why: Without clearly stating what you're trying to answer, data cleaning becomes arbitrary. This anchors the filtering decision in a real business objective.

Conclusion: Question defined — sets up the quantile analysis in Cell 11 to find the realistic price window scientifically.


In [ ]:
df['price'].quantile([
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99
])

What: Calculated price values at 9 different quantile levels — from bottom 1% to top 99%.

Why: Quantiles reveal where the realistic bulk of prices sit. The bottom 5% at $0 confirms many fake listings. The 99th percentile at $68,508 shows the realistic top of the market.

Conclusion: 50th percentile (median) = $17,900. 99th percentile = $68,508. Filtering to $0–$100,000 captures the real market range safely.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))

sns.histplot(
    df['price'],
    bins=100,
    kde=True
)

plt.title('Price Distribution')
plt.show()

What: Plotted histogram of raw price distribution with a KDE density curve.

Why: Visualizes just how extreme the skew is. When all bars cluster near zero and the x-axis stretches to $987M, the cleaning problem is impossible to ignore.

Conclusion: Histogram showed near-vertical spike at low prices with an invisible x-axis tail — visually confirmed the outlier problem before cleaning.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))

sns.histplot(
    np.log1p(df['price']),
    bins=100,
    kde=True
)

plt.title('Log Price Distribution')
plt.show()

What: Applied log transformation to price and re-plotted the distribution.

Why: Log transformation compresses extreme outliers and converts a skewed distribution into a bell curve — making it far easier to see the true shape of where most prices sit.

Conclusion: Log-transformed price showed a near-normal bell curve centered around 9–10 (equivalent to $8,000–$22,000). Confirmed log scaling is useful for modeling

In [ ]:
print("Original Shape:", df.shape)

clean_df = df[
    (df['price'] > 0) &
    (df['price'] <= 100000)
]

print("After Price Cleaning:", clean_df.shape)

print("Rows Removed:",
      df.shape[0] - clean_df.shape[0])

What: Filtered out all listings with price at or below $0 and above $100,000. Printed original shape, new shape, and rows removed.

Why: Fake, impossible, and extreme outlier prices corrupt model training. The model would waste capacity learning patterns from fake $987M listings instead of real car data.

Conclusion: Removed 972 rows. Dataset reduced from 18,400 to 17,428. Price range now $1–$100,000 — realistic and model-ready.


In [ ]:
clean_df['price'].describe()

What: Re-ran describe() on price after cleaning to verify the new distribution.

Why: Always verify your cleaning worked. Mean and max should now reflect realistic car prices, not distorted by fake listings.

Conclusion: Mean now $21,134. Max now $100,000. Std of $14,571 — a healthy spread. Cleaned price column looks like a real used car market dataset.


In [ ]:
print(clean_df['price'].skew())

What: Recalculated skewness of price after removing outliers.

Why: Confirms whether the outlier removal actually fixed the distribution problem. Lower skewness means the model's assumptions will be more valid.

Conclusion: Skewness dropped from 133.72 to 1.095 — a dramatic improvement. Price is now only moderately right-skewed, acceptable for regression modeling.


Feature Quality Assessment

Before EDA.

Before missing-value treatment.

Before correlation.

We must answer:

Which columns are actually useful for predicting price?

What: Markdown cell framing the next phase — evaluating every feature before deciding what to keep, fill, or drop.

Why: You cannot build a good model with bad features. Stating the evaluation question before touching the data prevents bias in feature selection decisions.

Conclusion: Mindset shift from cleaning to feature evaluation. Sets up Cells 18–20 for systematic assessment.

In [ ]:
missing_df = pd.DataFrame({
    'Missing_Count': clean_df.isnull().sum(),
    'Missing_%': (clean_df.isnull().sum()/len(clean_df))*100
})

missing_df = missing_df.sort_values(
    'Missing_%',
    ascending=False
)

display(missing_df)

What: Built a DataFrame showing missing count and missing percentage for every column, sorted worst to best.

Why: After price cleaning, null counts may have changed. Need an updated picture to decide exactly how to handle each column — drop, fill with 'unknown', or impute.

Conclusion: county = 100% missing (drop immediately). size = 72.8% missing (drop). condition = 41.2%, cylinders = 34.9%, drive = 28.1% — all fillable with 'unknown'.

In [ ]:
missing_df

What: Re-displayed the missing value summary table for clear viewing.

Why: Ensures the missing value table is clean and readable for documentation and decision-making reference.

Conclusion: Same findings as Cell 18 confirmed in clean tabular format — ready to drive drop/fill decisions in cleaning.


In [ ]:
clean_df.nunique().sort_values()

What: Counted unique values in every column, sorted from fewest to most.

Why: Columns with too many unique values (like model with thousands of car models) are problematic for encoding — they create hundreds of dummy columns that add noise.

Conclusion: transmission = 3 unique (clean). drive = 3, fuel = 5, condition = 6 (all usable). model had thousands of unique values — flagged for dropping before encoding

Feature Understanding

Before cleaning missing values.

Before encoding.

Before feature engineering.

We need to understand:

Numeric Features

In [ ]:
numeric_cols = [
    'price',
    'year',
    'odometer',
    'lat',
    'long'
]

What: Defined the list of columns that are genuinely numeric and meaningful for analysis.

Why: Separating numeric from categorical columns is the prerequisite for statistical analysis, correlation calculation, and appropriate preprocessing.

Conclusion: 5 true numeric features identified. lat and long included as geographic price signals. year and odometer are the primary car-condition predictors.


In [ ]:
clean_df[numeric_cols].describe().T

What: Transposed describe() for all 5 numeric columns — count, mean, min, max, quartiles in one readable table.

Why: Reveals data issues per feature — year with min of 1901 is suspicious, odometer with massive std suggests extreme spread, lat/long confirms geographic coverage.

Conclusion: year ranges 1901–2020. Odometer mean = 89,849 but std = 175,989 — massive spread indicating outliers. lat and long show nationwide US coverage.

In [ ]:
clean_df[numeric_cols].skew()

What: Calculated skewness for all 5 numeric features.

Why: Skewness above 1 or below -1 indicates a feature needs transformation or outlier treatment before regression modeling.

Conclusion: odometer skewness = 41.4 — extreme right skew. year = -3.47 (left-skewed, many old cars). price = 1.09 (manageable). lat/long near normal.


In [ ]:
clean_df['odometer'].quantile([
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99
])

What: Calculated odometer reading at 9 quantile levels from 1st to 99th percentile.

Why: Odometer is the #1 predictor of car price after age. Understanding its realistic range helps set a cleaning threshold — a car with 5M miles is a data error.

Conclusion: 99th percentile = 279,437 miles. Realistic cap of 300,000 miles established for filtering. Median = 74,068 miles — typical used car mileage.


In [ ]:
clean_df['year'].quantile([
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99
])

What: Calculated car year at 9 quantile levels.

Why: Year directly drives vehicle_age, which is a top price predictor. Need to know if extreme old years (1901) are outliers or legitimate vintage car listings.

Conclusion: 1st percentile = 1966. 5th percentile = 1996. 422 cars pre-1980 exist but are kept as legitimate vintage listings. 99th percentile = 2020 — no future-dated fraud.

In [ ]:
print("Cars before 1980:",
      (clean_df['year'] < 1980).sum())

print("Cars after 2022:",
      (clean_df['year'] > 2022).sum())

What: Counted cars manufactured before 1980 and after 2022.

Why: Cars after 2022 would be future-dated fraudulent entries. Cars before 1980 may be real vintage/classic listings worth keeping.

Conclusion: 422 cars pre-1980 (vintage — kept). 0 cars after 2022 (no fraud detected). Year column is clean without further filtering needed.

In [ ]:
clean_df = clean_df[
    (clean_df['odometer'] <= 300000)
]

What: Removed all rows with odometer readings above 300,000 miles.

Why: Cars with 500K–5M miles are almost certainly data entry errors. Including them would teach the model a false relationship between extreme mileage and price.

Conclusion: Rows with odometer above 300,000 removed. Dataset reduced to 17,187 rows. Mileage data now reflects realistic used car range.

In [ ]:
print(clean_df.shape)

What: Confirmed the new dataset size after odometer outlier removal.

Why: Always verify the impact of each filter. Helps track how many rows each cleaning step costs — ensuring you're not losing too much data.

Conclusion: 17,187 rows × 26 columns. Lost only 241 rows from odometer filtering — a small, acceptable cost for significantly cleaner data.


In [ ]:
print(clean_df['odometer'].skew())

What: Recalculated odometer skewness after removing rows above 300,000 miles.

Why: Confirms whether the outlier removal fixed the extreme skew problem. Skewness should have dropped dramatically.

Conclusion: Skewness dropped from 41.44 to 0.735 — near-normal distribution achieved. Odometer is now a well-behaved feature suitable for regression modeling.

In [ ]:
clean_df['vehicle_age'] = 2022 - clean_df['year']

clean_df['vehicle_age'].describe()

What: Created a new column vehicle_age by subtracting the car's manufacture year from 2022.

Why: year alone is hard for a model to interpret (what does "2015" mean?). vehicle_age = 7 years is a direct, intuitive predictor — older car means lower price.

Conclusion: vehicle_age created with mean of 10.4 years and range 0–121 years. Far more interpretable and predictive than raw year. This is the primary age feature used in modeling.

In [ ]:
clean_df['posting_date'].head(10)

What: Viewed the first 10 values of the posting_date column to understand its raw format.

Why: Dates stored as strings cannot be used mathematically. Need to understand the format before converting — wrong format specification would corrupt the conversion.

Conclusion: Dates stored as ISO 8601 format with timezone offset (e.g. 2021-05-04T12:31:18-0500). pd.to_datetime() with utc=True needed for clean conversion.


In [ ]:
clean_df['posting_date'] = pd.to_datetime(
    clean_df['posting_date'],
    utc=True,
    errors='coerce'
)

What: Converted the posting_date string column to a proper pandas datetime object with UTC timezone handling.

Why: String dates cannot be used for time-based feature extraction. Proper datetime format unlocks .dt.year, .dt.month, .dt.day — turning a text column into 3 numeric features.

Conclusion: Conversion successful — dtype changed to datetime64[ns, UTC]. Ready for feature extraction in Cell 35.

In [ ]:
print(clean_df['posting_date'].dtype)

What: Printed the data type of posting_date after conversion.

Why: Always verify type conversions worked. A failed conversion silently keeps the object dtype, breaking all downstream date operations.

Conclusion: dtype confirmed as datetime64[ns, UTC]. Conversion successful. Date-based features can now be safely extracted.

In [ ]:
clean_df['posting_year'] = clean_df['posting_date'].dt.year
clean_df['posting_month'] = clean_df['posting_date'].dt.month
clean_df['posting_day'] = clean_df['posting_date'].dt.day

What: Extracted year, month, and day from the datetime column into 3 separate numeric columns.

Why: ML models cannot use a datetime object directly. Breaking it into numeric components lets the model detect seasonal pricing patterns (e.g. cars priced higher in spring).

Conclusion: 3 new numeric features created — posting_year, posting_month, posting_day. Date information preserved in a form the model can actually use.

In [ ]:
clean_df[['posting_date',
          'posting_year',
          'posting_month',
          'posting_day']].head()

What: Displayed original posting_date alongside the 3 extracted features for visual verification.

Why: Verify that year/month/day were extracted correctly from the datetime. Timezone handling can sometimes shift dates by 1 day — this check catches that.

Conclusion: All 3 extracted correctly. May 4, 2021 → posting_year=2021, posting_month=5, posting_day=4. Extraction is accurate.


In [ ]:
print(clean_df['posting_year'].value_counts())

print(clean_df['posting_month'].value_counts().sort_index())

What: Checked how posting years and months are distributed across the dataset.

Why: If all listings are from one year, posting_year adds zero value as a feature (zero variance = zero predictive power).

Conclusion: All 17,187 listings from 2021 only — posting_year has zero variance, will be dropped. Only April (11,780) and May (5,407) present — posting_month has minimal value too.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample_df = clean_df.sample(10000, random_state=42)

plt.figure(figsize=(10,6))

sns.scatterplot(
    x='vehicle_age',
    y='price',
    data=sample_df,
    alpha=0.4
)

plt.title('Price vs Vehicle Age')
plt.show()

What: Plotted a scatter of 10,000 random listings showing price vs vehicle age.

Why: Visually confirms whether a linear or non-linear relationship exists. If older cars always cost less in a curved pattern, linear regression alone won't capture it well.

Conclusion: Clear downward trend confirmed — older cars cost less. Relationship is non-linear (curved), suggesting tree-based models will outperform linear regression here.

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    x='odometer',
    y='price',
    data=sample_df,
    alpha=0.4
)

plt.title('Price vs Odometer')
plt.show()

What: Plotted price against odometer reading for 10,000 sampled listings.

Why: Odometer is expected to be the strongest predictor. The scatter plot confirms the direction and shape of this relationship before including it in the model.

Conclusion: Strong negative relationship confirmed — higher mileage = lower price. Correlation of -0.52 (Cell 57) was clearly visible as a downward-sloping scatter.

In [ ]:
print("Duplicate Rows:", clean_df.duplicated().sum())

What: Checked for fully duplicate rows in the cleaned dataset.

Why: Duplicate car listings would train the model to over-learn those specific listings' prices, reducing generalization to new listings.

Conclusion: 0 duplicate rows — dataset is clean. No deduplication step needed. All records represent unique car listings.

In [ ]:
missing_df = pd.DataFrame({
    'Missing_Count': clean_df.isnull().sum(),
    'Missing_Percent':
        round((clean_df.isnull().sum()/len(clean_df))*100,2)
})

missing_df = missing_df.sort_values(
    'Missing_Percent',
    ascending=False
)

display(missing_df)

What: Recalculated missing values across all columns after price and odometer filtering.

Why: After removing rows, null counts change. Need a fresh view to make final decisions about which columns to fill vs drop before modeling.

Conclusion: county still 100% missing (drop). size = 73.2% (drop). condition = 41.3%, cylinders = 34.9% — will fill with 'unknown'. manufacturer = 4.3% — fill with mode.

In [ ]:
clean_df.dtypes

What: Printed the data type of every column in the dataset.

Why: Before encoding, need to know exactly which columns are object (text) type — those need One-Hot Encoding.

Numeric types go directly into the model.
Conclusion: id, price, year, odometer are numeric. manufacturer, model, condition, fuel, title_status, transmission, drive, type, paint_color are all object — all need encoding.

In [ ]:
cat_cols = [
    'condition',
    'cylinders',
    'fuel',
    'title_status',
    'transmission',
    'drive',
    'size',
    'type',
    'paint_color',
    'manufacturer'
]

for col in cat_cols:
    print("\n")
    print("="*50)
    print(col)
    print(clean_df[col].value_counts(dropna=False))

What: Printed value distribution for 10 categorical columns including NaN counts — condition, cylinders, fuel, title_status, transmission, drive, size, type, paint_color, manufacturer.

Why: Reveals which categories are dominant, which are rare, and confirms how many nulls exist per column — guiding fill strategy and whether rare categories need grouping.

Conclusion: condition: good (5,467) and excellent (3,326) dominate. cylinders: 6-cyl (4,236) and 4-cyl (3,461) most common. transmission: automatic dominates heavily.


Drop Immediately

These columns either have no predictive value or are too incomplete.

In [ ]:
drop_cols = [
    'id',
    'url',
    'region_url',
    'image_url',
    'county',
    'VIN',
    'size',
    'posting_year'
]

What: Defined the list of 8 columns to be dropped from the dataset.

Why: id and URLs are identifiers with no predictive signal. county is 100% null. VIN is a unique identifier per car — zero generalization. size is 73% missing. posting_year has zero variance.

Conclusion: 8 columns identified for removal. Dropping them reduces noise without losing any predictive signal.

In [ ]:
keep_cols = [
    'price',
    'year',
    'vehicle_age',
    'manufacturer',
    'model',
    'condition',
    'cylinders',
    'fuel',
    'odometer',
    'title_status',
    'transmission',
    'drive',
    'type',
    'paint_color',
    'region',
    'state',
    'lat',
    'long',
    'posting_month',
    'posting_day',
    'description'
]

What: Explicitly listed all columns to retain for modeling.

Why: Defining what to keep (not just what to drop) prevents accidental loss of important columns and makes the feature selection decision transparent and auditable.

Conclusion: 21 columns retained covering price, car specs, geography, and listing metadata. Clear, intentional feature set going into modeling.

In [ ]:
df_final = clean_df.copy()

# Drop useless columns
df_final = df_final.drop(
    columns=[
        'id',
        'url',
        'region_url',
        'image_url',
        'county',
        'VIN',
        'size',
        'posting_year'
    ]
)

# Fill categorical nulls
cat_fill = [
    'condition',
    'cylinders',
    'drive',
    'paint_color',
    'type'
]

for col in cat_fill:
    df_final[col] = df_final[col].fillna('unknown')

# Manufacturer
df_final['manufacturer'] = df_final['manufacturer'].fillna(
    df_final['manufacturer'].mode()[0]
)

# Drop tiny numerical nulls
df_final = df_final.dropna(
    subset=[
        'year',
        'vehicle_age',
        'lat',
        'long'
    ]
)

print(df_final.shape)

df_final.isnull().sum().sort_values(ascending=False)

What: Executed all cleaning decisions in one cell — dropped 8 junk columns, filled 5 categorical columns with 'unknown', filled manufacturer nulls with the most common value.

Why: Consolidating all cleaning into one cell makes it easy to re-run, debug, and verify. The output shape confirms everything worked as expected.

Conclusion: df_final created. Remaining nulls: title_status (225), model (194), fuel (135), transmission (54), description (1) — minor and handled in Cell 50.

In [ ]:
df_final.shape

What: Confirmed the shape of the final cleaned dataset.

Why: Quick sanity check after a major cleaning step. Shape tells you if any accidental row drops happened or if column drops went wrong.

Conclusion: 17,019 rows × 22 columns. Expected shape confirmed. Cleaning pipeline executed cleanly.

In [ ]:
df_final.isnull().sum().sort_values(ascending=False)

What: Checked remaining null values across all columns in df_final.

Why: After the cleaning pipeline, a few columns still have small null counts. Need to decide whether to fill or drop these remaining nulls.

Conclusion: title_status (225), model (194), fuel (135), transmission (54), description (1) still null. Small enough to drop those rows entirely in Cell 50.

In [ ]:
df_final = df_final.dropna()

print(df_final.shape)

df_final.isnull().sum().sum()

What: Dropped all remaining rows that had any null value in any column. Verified zero nulls remained.

Why: The remaining nulls (max 225 rows = 1.3% of data) are small enough that dropping is safer than imputing — imputing title_status or fuel with a guess could introduce false patterns.

Conclusion: Final dataset: 16,472 rows × 22 columns. Zero nulls remaining. Clean, complete dataset ready for EDA and modeling.

In [ ]:
df_final.to_csv(
    "used_car_cleaned.csv",
    index=False
)

print("Dataset Saved Successfully")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))

sns.histplot(
    df_final['price'],
    bins=50,
    kde=True
)

plt.title("Final Price Distribution")
plt.xlabel("Price")
plt.ylabel("Count")

plt.show()

What: Plotted the price distribution of the final cleaned dataset.

Why: After all cleaning, visualize the target variable one final time to confirm it looks like a realistic used car market — bell-shaped with a slight right tail.

Conclusion: Distribution confirmed realistic — centered around $15,000–$25,000, slight right skew, no extreme outliers. Model now has a clean, learnable target variable.

In [ ]:
top_makes = df_final['manufacturer'].value_counts().head(15).index

plt.figure(figsize=(14,6))

sns.boxplot(
    data=df_final[df_final['manufacturer'].isin(top_makes)],
    x='manufacturer',
    y='price'
)

plt.xticks(rotation=45)

plt.title("Price Distribution by Manufacturer")

plt.show()

What: Plotted price distributions for the top 15 manufacturers side by side using boxplots.

Why: Manufacturer is a major price driver. Boxplots show not just average prices but the spread and outliers per brand — a richer picture than averages alone.

Conclusion: Ferrari, Tesla, Porsche, RAM at the top. Ford, Chevrolet, Honda clustered in the middle. Clear price stratification by brand — manufacturer is a strong predictor.

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df_final,
    x='condition',
    y='price'
)

plt.xticks(rotation=45)

plt.title("Price vs Condition")

plt.show()

What: Plotted price distribution across 6 condition categories — new, like new, excellent, good, fair, salvage.

Why: Condition directly affects resale value. If condition clearly separates price ranges, it validates including it as a feature.

Conclusion: New and like new conditions have highest median prices. Salvage condition is lowest by a large margin. Condition is a valid and predictive feature.

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df_final,
    x='fuel',
    y='price'
)

plt.xticks(rotation=45)

plt.title("Price vs Fuel Type")

plt.show()

What: Compared price distributions across fuel types — gas, diesel, electric, hybrid, other.

Why: Fuel type affects both the car's value and its buyer market. Electric and diesel vehicles tend to command premiums — this should show up in price differences.

Conclusion: Diesel showed higher median prices. Electric also above average. Gas cars clustered in the middle. Fuel type carries meaningful pricing signal.

In [ ]:
plt.figure(figsize=(12,6))

sns.boxplot(
    data=df_final,
    x='type',
    y='price'
)

plt.xticks(rotation=45)

plt.title("Price vs Vehicle Type")

plt.show()

What: Plotted price by vehicle type — pickup, truck, SUV, coupe, sedan, convertible, van, wagon, mini-van, bus, etc.

Why: Vehicle type is one of the strongest categorical price drivers. Trucks and pickups consistently command higher prices than sedans.

Conclusion: Pickup and truck types have highest median prices ($31,998 and $26,435). Sedan and wagon at the lower end. Vehicle type confirmed as a strong predictor.

In [ ]:
import numpy as np

numeric_cols = [
    'price',
    'vehicle_age',
    'odometer',
    'year',
    'lat',
    'long',
    'posting_month',
    'posting_day'
]

corr = df_final[numeric_cols].corr()

corr['price'].sort_values(ascending=False)

What: Calculated Pearson correlation of all numeric features with price and sorted from highest to lowest.

Why: Correlation shows the linear relationship strength between each feature and the target. High correlation = feature is linearly predictive. Negative = inverse relationship (more miles = lower price).

Conclusion: odometer = -0.52 (strongest). year = +0.376, vehicle_age = -0.376 (same magnitude, opposite direction). lat and long show weak geographic correlation. These rankings guide feature importance expectations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))

sns.heatmap(
    df_final[numeric_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Numerical Feature Correlation')
plt.show()

What: Plotted a color-coded heatmap of correlations between all numeric features and price.

Why: Heatmap reveals not just feature-price relationships but also inter-feature correlations. If two features correlate strongly with each other (multicollinearity), one may be redundant.

Conclusion: year and vehicle_age are perfectly negatively correlated (-1.0) — they encode identical information. Keeping both would cause multicollinearity. year dropped

In [ ]:
df_model = df_final.copy()

df_model = df_model.drop(
    columns=[
        'year',
        'posting_date'
    ]
)

What: Dropped year and posting_date from the modeling dataset.

Why: year is perfectly captured by vehicle_age (correlation = -1.0). Keeping both is redundant and causes multicollinearity. posting_date is the raw datetime, replaced by posting_month and posting_day.

Conclusion: Dataset trimmed cleanly. vehicle_age retained as the more intuitive age representation. Both redundant columns eliminated.

In [ ]:
cat_cols = df_model.select_dtypes(
    include='object'
).columns

for col in cat_cols:
    print(f"{col}: {df_model[col].nunique()}")

What: Printed the number of unique values for every remaining categorical column.

Why: High-cardinality columns (like model with 4,052 unique values) create thousands of dummy columns after encoding — crashing memory and adding pure noise.

Conclusion: model = 4,052 unique values (drop — unmanageable). description = 14,305 unique (drop — free text). region = 22, manufacturer = 39, type = 14 — all manageable.

In [ ]:
drop_cols = [
    'description',
    'year',
    'posting_date'
]

In [ ]:
# Top manufacturers by average price

manufacturer_price = (
    df_model
    .groupby('manufacturer')['price']
    .mean()
    .sort_values(ascending=False)
)

print(manufacturer_price.head(15))

What: Calculated and ranked the average price per manufacturer across top 15 brands.

Why: Validates that manufacturer is a strong price predictor and reveals which brands command premiums — useful both for modeling and business interpretation.

Conclusion: Ferrari avg $98,900. Tesla $39,334. Porsche $37,928. Toyota and Honda in the $18,000–$21,000 range. Clear brand hierarchy confirmed — manufacturer encodes massive pricing signal.

In [ ]:
# Top vehicle types by average price

type_price = (
    df_model
    .groupby('type')['price']
    .mean()
    .sort_values(ascending=False)
)

print(type_price)

What: Calculated average price for each vehicle type — all 14 types ranked.

Why: Validates vehicle type as a strong predictor and quantifies the price gap between types for business understanding.

Conclusion: Pickup = $31,998 (highest).
Truck = $26,435. SUV = $20,591. Sedan = $15,970. Bus = $6,670 (lowest). $25,000+ gap between top and bottom types — vehicle type is a critical predictor.

In [ ]:
df_model = df_final.copy()

df_model['miles_per_year'] = (
    df_model['odometer'] /
    (df_model['vehicle_age'] + 1)
)

What: Created a new feature dividing total odometer reading by vehicle age (+1 to avoid division by zero).

Why: A 5-year-old car with 150K miles is used very differently than a 10-year-old car with the same mileage. miles_per_year captures usage intensity — a richer signal than mileage alone.

Conclusion: New feature created. Captures driving intensity per year — a car driven 30K miles/year degrades faster and prices lower than one driven 5K miles/year

In [ ]:
df_model['age_group'] = pd.cut(
    df_model['vehicle_age'],
    bins=[0,3,7,15,200],
    labels=[
        'new',
        'recent',
        'old',
        'very_old'
    ]
)

What: Bucketed vehicle_age into 4 categorical groups — new (0–3 years), recent (3–7), old (7–15), very old (15+).

Why: Price depreciation is not linear — a car drops sharply in its first 3 years, then more gradually. Binning captures these non-linear depreciation stages that a numeric age feature alone misses.

Conclusion: 4 age buckets created. Captures depreciation curve non-linearity. Paired with numeric vehicle_age gives the model both continuous and categorical age signal.

In [ ]:
df_model[['vehicle_age',
          'odometer',
          'miles_per_year',
          'age_group']].head()

What: Displayed first 5 rows of all 4 engineered features side by side for verification.

Why: Sanity check that calculations are correct. An 8-year-old car with 57,923 miles should show miles_per_year ≈ 6,436 — and it does.

Conclusion: All engineered features verified correct. vehicle_age, odometer, miles_per_year, and age_group all make logical sense row by row.

In [ ]:
print(df_model.shape)

In [ ]:
num_cols = [
    'price',
    'vehicle_age',
    'odometer',
    'miles_per_year'
]

corr = df_model[num_cols].corr()

display(corr)

What: Calculated correlation matrix for price and the 3 key numeric/engineered features.

Why: Validates that engineered features have meaningful correlation with price and checks for multicollinearity between them.

Conclusion: odometer–price = -0.52 (strongest). vehicle_age–price = -0.376. miles_per_year–price = -0.176. odometer and miles_per_year correlate 0.667 with each other — moderate but acceptable.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df_model,
    x='age_group',
    y='price'
)

plt.title('Price by Age Group')
plt.show()

What: Plotted price distribution across the 4 engineered age buckets — new, recent, old, very_old.

Why: Validates that the age_group feature actually separates price meaningfully. If the boxes overlap completely, the feature adds no value.

Conclusion: Clear price separation confirmed. new age group has highest median price. very_old lowest. Non-linear depreciation pattern visible — price drops fast for new→recent, slower for old→very_old.

In [ ]:
try:
    print("df_final:", df_final.shape)
except:
    print("df_final NOT FOUND")

try:
    print("clean_df:", clean_df.shape)
except:
    print("clean_df NOT FOUND")

try:
    print("df:", df.shape)
except:
    print("df NOT FOUND")

What: Verified which DataFrames are still in memory and their current shapes.

Why: In long notebooks, variables can get overwritten or lost. This diagnostic confirms which version of the data is currently active before re-loading.

Conclusion: df_final (16,472 × 22), clean_df (17,187 × 30), df (18,400 × 26) all in memory. Confirmed df_final is the clean version to use for modeling.

In [ ]:
import pandas as pd
import numpy as np

df_final = pd.read_csv("used_car_cleaned.csv")

print(df_final.shape)

In [ ]:
df_model = df_final.copy()

df_model['miles_per_year'] = (
    df_model['odometer'] /
    (df_model['vehicle_age'] + 1)
)

df_model['age_group'] = pd.cut(
    df_model['vehicle_age'],
    bins=[0,3,7,15,200],
    labels=['new','recent','old','very_old']
)

df_model.drop(
    columns=[
        'description',
        'posting_date',
        'year'
    ],
    inplace=True
)

print(df_model.shape)

What: Re-applied all feature engineering (miles_per_year, age_group) and dropped final junk columns (description, posting_date, year) on the reloaded clean dataset.

Why: Since the dataset was reloaded from CSV, all in-memory engineered features were lost. This cell rebuilds the full model-ready dataset in one clean block.

Conclusion: df_model created with 16,472 rows × 21 columns. All features engineered, all junk dropped. Full model dataset ready for X/y split

In [ ]:
X = df_model.drop('price', axis=1)
y = df_model['price']

print(X.shape)
print(y.shape)

What: Separated the dataset into features (X — all columns except price) and target (y — price column only).

Why: Every ML model requires this exact separation. X contains what the model learns from. y contains what it learns to predict. They must never overlap.

Conclusion: X shape: (16,472 × 20). y shape: (16,472,). Clean separation confirmed. Ready for encoding and train-test split.

In [ ]:
X = X.drop('model', axis=1)

print(X.shape)

What: Dropped the model column (specific car model name) from features before encoding.

Why: model has 4,052 unique values. One-Hot Encoding it would create 4,051 dummy columns — massive memory usage and near-zero signal per column. The manufacturer column already captures brand-level pricing.

Conclusion: X reduced to 19 features. model dropped cleanly. Encoding in next cell will now produce a manageable number of columns.

In [ ]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True,
    dtype='int'
)

print(X_encoded.shape)

What: Applied One-Hot Encoding to all remaining object/categorical columns using pd.get_dummies with drop_first=True to avoid multicollinearity.

Why: ML regression algorithms require all inputs to be numeric. Encoding converts categories like "Toyota" or "automatic" into 0/1 binary columns the model can calculate with.

Conclusion: Features expanded from 19 to 125 columns. 125 is manageable — no curse of dimensionality. dtype='int' ensures compact binary representation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

What: Split encoded data into 80% training (13,177 rows) and 20% testing (3,295 rows) with fixed random seed.

Why: Models must be evaluated on data they've never seen. random_state=42 ensures reproducible splits — same rows go to train/test every time the code runs.

Conclusion: X_train (13,177 × 125), X_test (3,295 × 125). No stratify needed for regression (stratify is for classification). Clean split ready.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

What: Trained a Linear Regression model on the training set and generated price predictions on the test set.

Why: Linear Regression is the simplest regression baseline — assumes a straight-line relationship between each feature and price. Sets the minimum performance bar all complex models must beat.

Conclusion: Model trained. Predictions generated. Evaluation in Cell 78 will reveal how well a linear assumption fits used car pricing.

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

r2 = r2_score(y_test, y_pred_lr)

mae = mean_absolute_error(y_test, y_pred_lr)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_lr)
)

print("Linear Regression Results")
print("="*40)

print("R²   :", round(r2,4))
print("MAE  :", round(mae,2))
print("RMSE :", round(rmse,2))

What: Evaluated Linear Regression using R², MAE, and RMSE — the 3 standard regression metrics.

Why: R² tells how much price variance the model explains. MAE tells the average prediction error in dollars. RMSE penalizes large errors more heavily — critical for pricing accuracy.

Conclusion: R² of 0.687 — model explains 68.7% of price variance. MAE of $5,655 means predictions are off by $5,655 on average. Decent baseline but significant room to improve with non-linear models.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    random_state=42,
    max_depth=15
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

What: Trained a Decision Tree Regressor with maximum depth of 15 on the training data.

Why: Decision Trees capture non-linear price relationships that Linear Regression misses — like "if odometer > 100K AND vehicle_age > 10 then price drops sharply." max_depth=15 balances learning vs overfitting.

Conclusion: Model trained. Tree can now make non-linear, rule-based price predictions. Evaluation in Cell 80 reveals improvement over linear baseline.

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

r2 = r2_score(y_test, y_pred_dt)

mae = mean_absolute_error(y_test, y_pred_dt)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_dt)
)

print("Decision Tree Results")
print("="*40)

print("R²   :", round(r2,4))
print("MAE  :", round(mae,2))
print("RMSE :", round(rmse,2))

What: Evaluated Decision Tree Regressor on R², MAE, and RMSE.

Why: Compare against Linear Regression baseline. Improvement in all 3 metrics confirms non-linear relationships exist in used car pricing that a linear model couldn't capture.

Conclusion: R² improved to 0.725. MAE dropped to $4,321 — $1,334 better than Linear Regression. RMSE also improved. Non-linear decision rules clearly help. But single trees can overfit — Random Forest next.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

What: Trained a Random Forest of 100 Decision Trees with max_depth=20 and parallel processing on all CPU cores.

Why: 100 trees each trained on random data and feature subsets then averaged — reduces the high variance of a single Decision Tree. Ensemble wisdom dramatically outperforms any single tree.

Conclusion: Model trained using full CPU parallelism. 100 trees will collectively produce far more stable and accurate price predictions than any single tree. Results in Cell 82.


In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

r2 = r2_score(y_test, y_pred_rf)

mae = mean_absolute_error(y_test, y_pred_rf)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_rf)
)

print("Random Forest Results")
print("="*40)

print("R²   :", round(r2,4))
print("MAE  :", round(mae,2))
print("RMSE :", round(rmse,2))

What: Evaluated Random Forest Regressor — the final and best model — on R², MAE, and RMSE.

Why: Random Forest is the strongest model tested. Its ensemble approach handles non-linearity, high-dimensional encoded features, and complex feature interactions better than any individual model.

Conclusion: R² = 0.84 — explains 84% of all used car price variance. MAE = $3,296 — predictions are on average only $3,296 off from actual price. RMSE = $5,677 — a $2,258 improvement over Linear Regression. Random Forest is the recommended model. Deploy this for used car price prediction.

In [ ]:
model_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'R² Score': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_dt),
        r2_score(y_test, y_pred_rf)
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_dt),
        mean_absolute_error(y_test, y_pred_rf)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_dt)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf))
    ]
})

model_comparison = model_comparison.sort_values(by='R² Score', ascending=False)
display(model_comparison.round(4))